In [ ]:
from nb_utils import set_root

PROJECT_ROOT = set_root(level=2)


# Beta4-IRT and CLAIRE for Latent-Ability-Aware Evaluation

This notebook extends the classical binary IRT story. The objective is to show how a Beta4-style response function and CLAIRE provide a more expressive view of model behavior across items with different latent properties.


## Learning Objectives

By the end of this notebook, students should be able to:

- inspect Beta4-style ICCs and compare them with classical binary IRT curves;
- interpret item difficulty and effective discrimination in a toy latent-ability setting;
- compare raw observed performance with a latent-aware summary;
- understand where CLAIRE fits in the workshop narrative.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from utils.handson import (
    make_item_bank,
    make_response_matrix,
    plot_beta4_family,
    plot_iccs,
    plot_latent_accuracy,
    simulate_latent_ability_dataset,
    summarize_latent_results,
)


## Step 1: Create a small Beta4 item bank

The item bank below contains a toy set of items with different latent difficulties and discrimination components. This is enough to illustrate how the response curves change.


In [ ]:
item_bank = make_item_bank()
item_bank


In [ ]:
ax = plot_iccs(item_bank)
plt.show()


## Step 2: Inspect the Beta4 family

In this plot we keep difficulty fixed and vary the discrimination components. This helps explain why Beta4 is useful when we want more flexibility than a basic binary IRT curve.


In [ ]:
ax = plot_beta4_family()
plt.show()


## Step 3: Simulate a latent-ability evaluation setting

We now simulate several models answering the same item bank. In this toy example, we know the latent quantities used to generate the data.

For the workshop, that is helpful because students can first learn the interpretation before discussing estimation.


In [ ]:
responses = simulate_latent_ability_dataset(n_models=6, item_bank=item_bank, random_state=21)
responses.head(9)


In [ ]:
response_matrix = make_response_matrix(responses)
response_matrix


In [ ]:
latent_summary = summarize_latent_results(responses)
latent_summary.round(3)


In [ ]:
claire_ready = (
    responses.groupby('model', as_index=False)
    .agg(
        observed_accuracy=('observed_correct', 'mean'),
        expected_probability=('expected_probability', 'mean'),
        mean_effective_discrimination=('effective_discrimination', 'mean'),
        mean_difficulty_probability=('difficulty_probability', 'mean'),
    )
    .sort_values('expected_probability', ascending=False)
)

claire_ready.round(3)


In [ ]:
ax = plot_latent_accuracy(latent_summary)
plt.show()


## Where CLAIRE fits

A concise way to explain CLAIRE in this notebook is:

1. standard evaluation gives aggregate correctness;
2. IRT gives a language for latent ability and item difficulty;
3. Beta4 gives a richer response family;
4. CLAIRE uses this latent perspective to evaluate supervised methods more informatively.

In this toy notebook, the tables above act as a **CLAIRE-ready view** of the data: models, item difficulty, discrimination, and success behavior are all visible in the same evaluation story.


## Activity: Beta4 and CLAIRE Interpretation

Ask students to answer the following:

1. Which Beta4 item looks hardest, and how do you justify that from the item bank and ICC plot?
2. Which item has the highest effective discrimination, and what kind of separation does that create?
3. Compare `observed_accuracy` and `expected_probability` in the summary table. Which model looks strongest under each view?
4. Why is this latent-aware view more informative than reporting a single metric alone?


In [ ]:
# Student workspace
# Suggested mini-task:
# Rank the models by observed accuracy and by expected probability.
# Then discuss whether the rankings tell the same story.

claire_ready[['model', 'observed_accuracy', 'expected_probability']].sort_values(
    'observed_accuracy', ascending=False
)


## Suggested oral closing

The workshop can end this notebook with the following message: **standard metrics summarize outcomes, but latent-ability-aware methods explain structure**. That is the core motivation for discussing CLAIRE in this setting.
